# AI Job Displacement Analysis

Data Pipeline

# Imports

In [12]:
import os
import json
import pandas as pd
import numpy as np

# Target Occupations

In [13]:
TARGET_OCCUPATIONS = {
    "15-1252.00": {"title": "Software Developers", "category": "STEM / High-Cognitive / Digital"},
    "15-2051.00": {"title": "Data Scientists", "category": "STEM / High-Cognitive / Digital"},
    "29-1141.00": {"title": "Registered Nurses", "category": "Healthcare / High-Empathy / Physical"},
    "29-2034.00": {"title": "Radiologic Technologists", "category": "Healthcare / Technical / Physical"},
    "29-1051.00": {"title": "Pharmacists", "category": "Healthcare / High-Precision / Regulated"},
    "25-2021.00": {"title": "Elementary School Teachers", "category": "Education / Social / Low-Predictability"},
    "23-1011.00": {"title": "Lawyers", "category": "Legal / High-Cognitive / Regulated"},
    "23-2011.00": {"title": "Paralegals and Legal Assistants", "category": "Legal / Administrative / Information"},
    "13-2051.00": {"title": "Financial Analysts", "category": "Finance / Quantitative / Digital"},
    "43-4051.00": {"title": "Customer Service Representatives", "category": "Admin / High-Routine / Social"},
    "43-6013.00": {"title": "Medical Secretaries", "category": "Admin / Routine / Regulated"},
    "53-3032.00": {"title": "Heavy and Tractor-Trailer Truck Drivers", "category": "Logistics / Physical / Medium-Predictability"},
    "47-2111.00": {"title": "Electricians", "category": "Trades / Physical-Complex / Dynamic-Environment"},
    "47-2152.00": {"title": "Plumbers, Pipefitters, and Steamfitters", "category": "Trades / Physical-Complex / Dynamic-Environment"},
    "47-2061.00": {"title": "Construction Laborers", "category": "Trades / Physical-Manual / Chaotic-Environment"},
    "35-1011.00": {"title": "Chefs and Head Cooks", "category": "Hospitality / Physical / Creative-Routine"},
    "35-3031.00": {"title": "Waiters and Waitresses", "category": "Hospitality / Physical / High-Social"},
    "27-1024.00": {"title": "Graphic Designers", "category": "Creative / Digital / Heuristic"},
    "11-2021.00": {"title": "Marketing Managers", "category": "Management / Creative / Strategic"},
    "17-2051.00": {"title": "Civil Engineers", "category": "Engineering / Physical-Digital / High-Consequence"},
    "51-4121.00": {"title": "Welders, Cutters, Solderers, and Brazers", "category": "Manufacturing / Physical-Precision"},
    "37-2011.00": {"title": "Janitors and Cleaners", "category": "Services / Physical-Manual / Routine"},
    "41-9022.00": {"title": "Real Estate Sales Agents", "category": "Sales / High-Social / High-Arbitrage"},
    "41-9041.00": {"title": "Telemarketers", "category": "Sales / High-Routine / Purely-Digital"},
    "33-9032.00": {"title": "Security Guards", "category": "Security / Physical-Observation / Low-Routine"}
}

TARGET_CODES = list(TARGET_OCCUPATIONS.keys())
print(f"Loaded {len(TARGET_CODES)} target SOC codes across diverse industries.")

Loaded 25 target SOC codes across diverse industries.


# Load Data

In [14]:
DATA_DIR = "../data/raw"

def load_csv_file(filename):
    filepath = os.path.join(DATA_DIR, filename)
    if not os.path.exists(filepath):
        raise FileNotFoundError(f"Missing file: {filepath}")
    try:
        return pd.read_csv(filepath)
    except Exception:
        return pd.read_csv(filepath, sep="\t")

def load_excel_sheet_1(filename):
    filepath = os.path.join(DATA_DIR, filename)
    if not os.path.exists(filepath):
        raise FileNotFoundError(f"Missing file: {filepath}")
    # Load only the first sheet (sheet_name=0)
    return pd.read_excel(filepath, sheet_name=0)

# Load all datasets
df_occ = load_csv_file("occupation_data.csv")
df_tasks = load_csv_file("task_statements.csv")
df_ratings = load_csv_file("task_ratings.csv")
df_context = load_csv_file("work_context.csv")
df_activities = load_csv_file("work_activities.csv")
df_skills = load_csv_file("essential_skills.csv")
df_software = load_csv_file("software_skills.csv")
df_wages = load_excel_sheet_1("wages.xlsx")

print("--- All Datasets Loaded Successfully ---")
print("Occupations Shape:", df_occ.shape)
print("Task Statements Shape:", df_tasks.shape)
print("Work Context Shape:", df_context.shape)
print("Software Skills Shape:", df_software.shape)
print("Wages Excel (Sheet 1) Shape:", df_wages.shape)

--- All Datasets Loaded Successfully ---
Occupations Shape: (1016, 3)
Task Statements Shape: (18796, 8)
Work Context Shape: (297676, 16)
Software Skills Shape: (31821, 7)
Wages Excel (Sheet 1) Shape: (413527, 32)


# SOC Code & Filter Datasets

In [15]:
def find_soc_col(df):
    """Finds the column containing SOC codes dynamically."""
    for col in df.columns:
        cl = col.lower()
        if "O*net-soc code" in cl or "occ_code" in cl:
            return col
    return df.columns[0]

def clean_soc(val):
    """Normalizes SOC format across O*NET and BLS wage files (e.g., '15-1252.00' -> '15-1252')."""
    if pd.isna(val):
        return ""
    s = str(val).strip()
    return s.split(".")[0]

# Standardize codes for 25 target occupations
TARGET_BASE_CODES = [clean_soc(c) for c in TARGET_CODES]

# Filter datasets
filtered_dfs = {}
raw_dfs = {
    "occupation": df_occ,
    "tasks": df_tasks,
    "ratings": df_ratings,
    "context": df_context,
    "activities": df_activities,
    "skills": df_skills,
    "software": df_software,
    "wages": df_wages
}

for name, df in raw_dfs.items():
    soc_col = find_soc_col(df)
    df["_norm_soc"] = df[soc_col].apply(clean_soc)
    filtered_dfs[name] = df[df["_norm_soc"].isin(TARGET_BASE_CODES)].copy()
    print(f"Filtered {name:12s}: {len(filtered_dfs[name])} matching records")

Filtered occupation  : 34 matching records
Filtered tasks       : 777 matching records
Filtered ratings     : 6579 matching records
Filtered context     : 10816 matching records
Filtered activities  : 2624 matching records
Filtered skills      : 640 matching records
Filtered software    : 2234 matching records
Filtered wages       : 19264 matching records


# Aggregate Occupation-Level Metadata

(Wages, Tools, Skills)

In [16]:
occ_summary = {}

for full_code, meta in TARGET_OCCUPATIONS.items():
    base_code = clean_soc(full_code)
    
    # Wages Data: Target 'A_Median' and calculate average across matching rows
    wage_sub = filtered_dfs["wages"][filtered_dfs["wages"]["_norm_soc"] == base_code]
    median_col = next((col for col in wage_sub.columns if col.lower() == "a_median"), None)
    
    if not wage_sub.empty and median_col is not None:
        # Coerce non-numeric markers (e.g., '*', '#', None) to NaN before taking mean
        wage_numeric = pd.to_numeric(wage_sub[median_col], errors="coerce")
        mean_wage = wage_numeric.mean()
        median_wage = round(float(mean_wage), 2) if pd.notnull(mean_wage) else "N/A"
    else:
        median_wage = "N/A"
    
    # Software Skills: Extract top entries targeting 'Element Name'
    soft_sub = filtered_dfs["software"][filtered_dfs["software"]["_norm_soc"] == base_code]
    if not soft_sub.empty and "Element Name" in soft_sub.columns:
        top_software = soft_sub["Element Name"].dropna().unique()[:5].tolist()
    elif not soft_sub.empty:
        # Fallback to first text column if 'Element Name' is absent in specific release
        top_software = soft_sub[soft_sub.columns[0]].dropna().unique()[:5].tolist()
    else:
        top_software = []
    
    # Essential Skills: Extract top entries targeting 'Element Name'
    skills_sub = filtered_dfs["skills"][filtered_dfs["skills"]["_norm_soc"] == base_code]
    if not skills_sub.empty and "Element Name" in skills_sub.columns:
        top_skills = skills_sub["Element Name"].dropna().unique()[:5].tolist()
    elif not skills_sub.empty:
        top_skills = skills_sub[skills_sub.columns[0]].dropna().unique()[:5].tolist()
    else:
        top_skills = []

    occ_summary[full_code] = {
        "median_wage": median_wage,
        "top_software_tools": top_software,
        "top_essential_skills": top_skills
    }

print("Sample Aggregated Occupation Metadata (Software Developers):")
print(json.dumps(occ_summary["15-1252.00"], indent=2))

Sample Aggregated Occupation Metadata (Software Developers):
{
  "median_wage": 121224.27,
  "top_software_tools": [
    "Word processing software",
    "Development environment software",
    "Object or component oriented development software",
    "Data base user interface and query software",
    "Document management software"
  ],
  "top_essential_skills": [
    "Reading Comprehension",
    "Active Listening",
    "Writing",
    "Speaking",
    "Mathematics"
  ]
}


# Clean Tasks & Merge Task Importance Rating

In [17]:
tasks_df = filtered_dfs["tasks"].copy()
ratings_df = filtered_dfs["ratings"].copy()

# Column detection
task_id_col = [c for c in tasks_df.columns if "task" in c.lower() and "id" in c.lower()][0]
task_text_col = [c for c in tasks_df.columns if "statement" in c.lower() or "task" in c.lower() and "id" not in c.lower()][0]

# Clean task statements
tasks_df["cleaned_task"] = (
    tasks_df[task_text_col]
    .astype(str)
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
)

# Extract Importance Rating ('IM')
im_ratings = ratings_df[ratings_df["Scale ID"] == "IM"].copy() if "Scale ID" in ratings_df.columns else ratings_df.copy()
val_col = "Data Value" if "Data Value" in im_ratings.columns else im_ratings.columns[-2]

# Merge tasks with importance rating
merged_tasks = pd.merge(
    tasks_df,
    im_ratings[["_norm_soc", task_id_col, val_col]],
    on=["_norm_soc", task_id_col],
    how="left"
).rename(columns={val_col: "task_importance"})

print(f"Merged {len(merged_tasks)} tasks with importance ratings.")
merged_tasks[["_norm_soc", task_id_col, "cleaned_task", "task_importance"]].head()

Merged 777 tasks with importance ratings.


,_norm_soc,Task ID,cleaned_task,task_importance
0,11-2021,951,"Identify, develop, or evaluate marketing strat...",4.30
1,11-2021,20709,"Formulate, direct, or coordinate marketing act...",4.24
2,11-2021,952,Evaluate the financial aspects of product deve...,3.99
3,11-2021,950,"Develop pricing strategies, balancing firm obj...",3.87
4,11-2021,957,Compile lists describing product or service of...,3.70


# Unified Dataset & Export

In [19]:
unified_records = []
flat_rows = []

df_occ_filtered = filtered_dfs["occupation"]
occ_desc_col = [c for c in df_occ_filtered.columns if "description" in c.lower()][0]

for soc_code, meta in TARGET_OCCUPATIONS.items():
    base_code = clean_soc(soc_code)
    
    # Extract job description
    occ_row = df_occ_filtered[df_occ_filtered["_norm_soc"] == base_code]
    job_desc = occ_row[occ_desc_col].values[0] if not occ_row.empty else ""
    
    # Extract occupation metadata aggregated from auxiliary files
    aux_meta = occ_summary.get(soc_code, {})
    
    # Extract job tasks
    job_tasks = merged_tasks[merged_tasks["_norm_soc"] == base_code]
    
    task_list = []
    for _, r in job_tasks.iterrows():
        t_id = int(r[task_id_col]) if pd.notnull(r[task_id_col]) else None
        t_desc = r["cleaned_task"]
        t_imp = round(float(r["task_importance"]), 2) if pd.notnull(r["task_importance"]) else None
        
        task_list.append({
            "task_id": t_id,
            "task_description": t_desc,
            "importance": t_imp
        })
        
        # Row for flat CSV batch LLM processing
        '''
        flat_rows.append({
            "soc_code": soc_code,
            "job_title": meta["title"],
            "sampling_category": meta["category"],
            "job_description": job_desc,
            "median_wage": aux_meta.get("median_wage", "N/A"),
            "top_software_tools": ", ".join(aux_meta.get("top_software_tools", [])),
            "top_essential_skills": ", ".join(aux_meta.get("top_essential_skills", [])),
            "task_id": t_id,
            "task_description": t_desc,
            "task_importance": t_imp
        })
        '''

    unified_records.append({
        "soc_code": soc_code,
        "job_title": meta["title"],
        "sampling_category": meta["category"],
        "job_description": job_desc,
        "occupation_metadata": aux_meta,
        "total_tasks_count": len(task_list),
        "tasks": task_list
    })

# Save Unified Nested JSON
json_out = "../data/processed/unified_data.json"
with open(json_out, "w", encoding="utf-8") as f:
    json.dump(unified_records, f, indent=2)

# Save Flat Task CSV
#csv_out = "../data/processed/unified_data.csv"
#df_flat = pd.DataFrame(flat_rows)
#df_flat.to_csv(csv_out, index=False)

print(f"SUCCESS: Exported JSON to '{json_out}'")
#print(f"SUCCESS: Exported flat CSV to '{csv_out}' ({len(df_flat)} total tasks)")

SUCCESS: Exported JSON to '../data/processed/unified_data.json'
